# Breast Ultrasound: Segmentation-Guided Radiomics/CNN Fusion

End-to-end pipeline for benign vs. malignant classification of breast ultrasound lesions,
trained on BUSI and externally validated on BUS-BRA.

```
BUSI ── pHash dedup ──┬─► UNet++ segmentation (Dice / IoU)
                      └─► predicted-mask ROI ─┬─► radiomics ─────┐
                                              └─► ResNet-50 ─────┴─► LASSO + LR ─► CV, DeLong, SHAP
BUS-BRA ──────────────────────────────────────────────────────────────────────────► external test
```

**Design decisions**
- Near-duplicates are removed by perceptual hashing *before* any split. BUSI has no patient
  IDs, so splits are image-level after dedup.
- All classification ROIs come from **predicted** masks (no manual annotation at inference).
  A reference-mask arm is kept as an upper-bound ablation.
- Scaling, LASSO selection and the classifier are fit inside each CV fold.
- The external cohort is never seen by either the segmenter or the classifiers.

Data layout and download links are in the repository `README.md`.

## 1. Setup

In [ ]:
import glob
import json
import os
import platform
import random
import re
import warnings

import albumentations as A
import cv2
import imagehash
import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import shap
import skimage
import sklearn
import torch
import torch.nn as nn
import torchvision.models as tvm
from PIL import Image
from scipy import ndimage
from scipy.stats import kurtosis, norm, skew
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import label as sklabel, regionprops
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = os.environ.get("DATA_DIR", "data")

CONFIG = {
    "busi_root": os.path.join(DATA_DIR, "Dataset_BUSI_with_GT"),
    "busbra_root": os.path.join(DATA_DIR, "BUS_BRA"),
    "busbra_id_col": "ID",
    "busbra_label_col": "Pathology",
    "results_dir": "results",
    "img_size": 256,
    "dedup_max_hamming": 4,
    "test_frac": 0.2,
    "val_frac": 0.15,
    # segmentation
    "arch": "UnetPlusPlus",
    "encoder": "efficientnet-b3",
    "epochs": 40,
    "batch_size": 8,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "seg_loss": "focal_tversky",  # or "dice_focal"
    "tversky_alpha": 0.3,
    "tversky_beta": 0.7,
    "tversky_gamma": 1.33,
    "seg_tta": True,
    "force_retrain_seg": False,
    # classification
    "cnn_backbone": "resnet50",
    "roi_pad": 0.15,
    "n_folds": 5,
    "internal_use_pred_mask": True,
    "ext_use_pred_mask": True,
    "force_rebuild_features": False,
}

RES = CONFIG["results_dir"]
os.makedirs(RES, exist_ok=True)
print(f"device={DEVICE}  results={RES}")

## 2. BUSI indexing and deduplication

Multi-part masks (`*_mask_1.png`) are merged. BUSI contains near-duplicate frames that leak
across naive splits; we keep one image per pHash cluster (64-bit DCT hash, Hamming ≤ 4).

In [ ]:
def index_busi(root):
    rows = []
    for cls in ["benign", "malignant", "normal"]:
        d = os.path.join(root, cls)
        if not os.path.isdir(d):
            continue
        for ip in sorted(f for f in glob.glob(os.path.join(d, "*.png")) if "_mask" not in f):
            base = os.path.splitext(os.path.basename(ip))[0]
            masks = sorted(glob.glob(os.path.join(d, f"{base}_mask*.png")))
            rows.append({"image": ip, "masks": masks, "class": cls})
    if not rows:
        raise FileNotFoundError(f"No BUSI images under {root}")
    return pd.DataFrame(rows)


def load_image_gray(path, size):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise FileNotFoundError(path)
    return cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)


def load_merged_mask(mask_paths, size):
    acc = np.zeros((size, size), np.uint8)
    for mp in mask_paths:
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        acc = np.maximum(acc, cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST))
    return (acc > 127).astype(np.uint8)


def dedup(df, max_hamming):
    hashes = [imagehash.phash(Image.open(p).convert("L"), hash_size=8)
              for p in tqdm(df["image"], desc="hashing")]
    keep, removed = [], []
    for i, h in enumerate(hashes):
        match = next((k for k in keep if h - hashes[k] <= max_hamming), None)
        if match is None:
            keep.append(i)
        else:
            removed.append({"removed_image": df["image"][i],
                            "kept_representative": df["image"][match],
                            "hamming_distance": h - hashes[match]})
    return df.iloc[keep].reset_index(drop=True), pd.DataFrame(removed)


busi_raw = index_busi(CONFIG["busi_root"])
busi, dup_log = dedup(busi_raw, CONFIG["dedup_max_hamming"])
dup_log.to_csv(os.path.join(RES, "busi_near_duplicates_removed.csv"), index=False)
print(f"dedup: {len(busi_raw)} -> {len(busi)} ({len(dup_log)} near-duplicates removed)")
print(busi["class"].value_counts().to_string())

## 3. Segmentation data

Segmentation and classification use benign + malignant images (normal masks are empty).
The split is made once and the held-out test ids are reused downstream.

In [ ]:
seg_df = busi[busi["class"] != "normal"].reset_index(drop=True)
labels = (seg_df["class"] == "malignant").astype(int)

trainval_idx, test_idx = train_test_split(
    seg_df.index, test_size=CONFIG["test_frac"], stratify=labels, random_state=SEED)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=CONFIG["val_frac"], stratify=labels[trainval_idx],
    random_state=SEED)
print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)

TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=20, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.RandomGamma(p=0.3),
    A.GaussNoise(p=0.2),
])


def to_tensor_input(img3_uint8):
    x = (img3_uint8.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return x.transpose(2, 0, 1).astype(np.float32)


class SegDataset(Dataset):
    def __init__(self, df, idx, size, augment=False):
        self.df, self.idx, self.size, self.augment = df, list(idx), size, augment

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        r = self.df.loc[self.idx[i]]
        img = np.repeat(load_image_gray(r["image"], self.size)[..., None], 3, axis=2)
        msk = load_merged_mask(r["masks"], self.size)
        if self.augment:
            out = TRAIN_AUG(image=img, mask=msk)
            img, msk = out["image"], out["mask"]
        return (torch.from_numpy(to_tensor_input(img)),
                torch.from_numpy((msk > 0).astype(np.float32)[None]))


train_loader = DataLoader(SegDataset(seg_df, train_idx, CONFIG["img_size"], augment=True),
                          batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(SegDataset(seg_df, val_idx, CONFIG["img_size"]),
                        batch_size=CONFIG["batch_size"])

## 4. UNet++ training

Focal-Tversky with β > α penalises missed lesion pixels more than false positives, which
targets the small-lesion failure mode. The best checkpoint by validation Dice is cached.

In [ ]:
def build_seg_loss(cfg):
    if cfg["seg_loss"] == "focal_tversky":
        return smp.losses.TverskyLoss(mode="binary", alpha=cfg["tversky_alpha"],
                                      beta=cfg["tversky_beta"], gamma=cfg["tversky_gamma"])
    dice, focal = smp.losses.DiceLoss(mode="binary"), smp.losses.FocalLoss(mode="binary")
    return lambda out, y: dice(out, y) + focal(out, y)


def batch_dice(logits, target, eps=1e-6):
    p = (torch.sigmoid(logits) > 0.5).float()
    inter = (p * target).sum((1, 2, 3))
    return ((2 * inter + eps) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps)).mean().item()


seg_model = getattr(smp, CONFIG["arch"])(
    encoder_name=CONFIG["encoder"], encoder_weights="imagenet",
    in_channels=3, classes=1).to(DEVICE)
seg_ckpt = os.path.join(RES, "unet_best.pt")
seg_history = []

if os.path.exists(seg_ckpt) and not CONFIG["force_retrain_seg"]:
    print(f"loading cached weights from {seg_ckpt}")
else:
    criterion = build_seg_loss(CONFIG)
    opt = torch.optim.AdamW(seg_model.parameters(), lr=CONFIG["lr"],
                            weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["epochs"])
    best = 0.0
    for ep in range(CONFIG["epochs"]):
        seg_model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            criterion(seg_model(x), y).backward()
            opt.step()
        sched.step()
        seg_model.eval()
        with torch.no_grad():
            vd = float(np.mean([batch_dice(seg_model(x.to(DEVICE)), y.to(DEVICE))
                                for x, y in val_loader]))
        seg_history.append(vd)
        if vd > best:
            best = vd
            torch.save(seg_model.state_dict(), seg_ckpt)
        print(f"epoch {ep + 1:02d}/{CONFIG['epochs']}  val_dice={vd:.3f}  "
              f"lr={sched.get_last_lr()[0]:.2e}")
    print(f"best val dice: {best:.3f}")

seg_model.load_state_dict(torch.load(seg_ckpt, map_location=DEVICE))
seg_model.eval();

## 5. Inference and held-out segmentation metrics

Four-way flip TTA, then hole filling and removal of speckle components (< 0.2% of the
image). These are exactly the masks used for feature extraction downstream.

In [ ]:
def clean_mask(m, min_frac=0.002):
    m = ndimage.binary_fill_holes(m).astype(np.uint8)
    lbl, n = ndimage.label(m)
    if n == 0:
        return m
    sizes = ndimage.sum(m, lbl, range(1, n + 1))
    keep_ids = [i for i, s in enumerate(sizes, 1) if s >= min_frac * m.size]
    return np.isin(lbl, keep_ids).astype(np.uint8)


@torch.no_grad()
def predict_mask(img_gray_uint8, size):
    im = cv2.resize(img_gray_uint8, (size, size))
    x = torch.from_numpy(to_tensor_input(np.repeat(im[..., None], 3, axis=2)))[None].to(DEVICE)
    flips = [[], [3], [2], [2, 3]] if CONFIG["seg_tta"] else [[]]
    probs = []
    for f in flips:
        out = seg_model(torch.flip(x, dims=f) if f else x)
        probs.append(torch.sigmoid(torch.flip(out, dims=f) if f else out))
    p = torch.stack(probs).mean(0)[0, 0].cpu().numpy()
    return clean_mask((p > 0.5).astype(np.uint8))


def dice_iou(gt, pr, eps=1e-6):
    inter = float((gt * pr).sum())
    total = float(gt.sum() + pr.sum())
    return (2 * inter + eps) / (total + eps), (inter + eps) / (total - inter + eps)


scores = []
for i in test_idx:
    row = seg_df.loc[i]
    img = load_image_gray(row["image"], CONFIG["img_size"])
    gt = load_merged_mask(row["masks"], CONFIG["img_size"])
    scores.append(dice_iou(gt, predict_mask(img, CONFIG["img_size"])))
test_dice, test_iou = map(np.array, zip(*scores))

q1, q3 = np.percentile(test_dice, [25, 75])
print(f"Dice  mean={test_dice.mean():.3f}  median={np.median(test_dice):.3f}  "
      f"IQR=[{q1:.3f}, {q3:.3f}]")
print(f"IoU   mean={test_iou.mean():.3f}  median={np.median(test_iou):.3f}")
print(f"failures (Dice < 0.2): {(test_dice < 0.2).sum()}/{len(test_dice)}")

## 6. Feature extraction

The ROI is the mask bounding box padded by 15%. Radiomics uses PyRadiomics when installed,
otherwise an equivalent scikit-image set (first-order, GLCM, 2D shape). Deep features are
the 2048-d pooled output of an ImageNet ResNet-50. Both extractors are frozen and
label-independent, so precomputing them once introduces no leakage into CV.

In [ ]:
def bbox_from_mask(mask, pad_frac):
    ys, xs = np.where(mask > 0)
    h, w = mask.shape
    if len(xs) == 0:
        return 0, 0, w, h
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    pw, ph = int((x1 - x0) * pad_frac), int((y1 - y0) * pad_frac)
    return max(0, x0 - pw), max(0, y0 - ph), min(w, x1 + pw), min(h, y1 + ph)


def crop_roi(img, mask, pad_frac):
    x0, y0, x1, y1 = bbox_from_mask(mask, pad_frac)
    return img if x1 <= x0 or y1 <= y0 else img[y0:y1, x0:x1]


try:
    import SimpleITK as sitk
    from radiomics import featureextractor

    _rad = featureextractor.RadiomicsFeatureExtractor()
    _rad.settings.update({"binWidth": 25, "force2D": True, "force2Ddimension": 0,
                          "label": 1, "normalize": True})
    _rad.disableAllFeatures()
    for fc in ["firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm", "shape2D"]:
        _rad.enableFeatureClassByName(fc)
    USE_PYRADIOMICS = True
except ImportError:
    USE_PYRADIOMICS = False
print("radiomics backend:", "pyradiomics" if USE_PYRADIOMICS else "scikit-image")


def radiomics_skimage(img, mask):
    vals = img[mask > 0]
    if vals.size == 0:
        vals = img.ravel()
    f = {"fo_mean": vals.mean(), "fo_std": vals.std(), "fo_skew": skew(vals),
         "fo_kurt": kurtosis(vals), "fo_energy": float((vals.astype(np.float64) ** 2).sum()),
         "fo_p10": np.percentile(vals, 10), "fo_p90": np.percentile(vals, 90),
         "fo_median": np.median(vals)}
    glcm = graycomatrix((img // 32).astype(np.uint8), [1], [0, np.pi / 2], levels=8,
                        symmetric=True, normed=True)
    for prop in ["contrast", "dissimilarity", "homogeneity", "ASM", "correlation"]:
        f[f"glcm_{prop}"] = float(graycoprops(glcm, prop).mean())
    lbl = sklabel(mask > 0)
    if lbl.max() >= 1:
        rp = max(regionprops(lbl), key=lambda r: r.area)
        f.update({"shp_area": rp.area, "shp_perimeter": rp.perimeter,
                  "shp_eccentricity": rp.eccentricity, "shp_solidity": rp.solidity,
                  "shp_extent": rp.extent})
    return f


def extract_radiomics(img_u8, mask_u8):
    if USE_PYRADIOMICS:
        try:
            res = _rad.execute(sitk.GetImageFromArray(img_u8.astype(np.float32)),
                               sitk.GetImageFromArray(mask_u8.astype(np.int16)))
            return {k: float(v) for k, v in res.items()
                    if k.startswith("original_") and np.isscalar(v)}
        except Exception:
            pass
    return radiomics_skimage(img_u8, mask_u8)


def build_backbone(name):
    m = getattr(tvm, name)(weights="IMAGENET1K_V2")
    m.fc = nn.Identity()
    return m.eval().to(DEVICE)


cnn = build_backbone(CONFIG["cnn_backbone"])


@torch.no_grad()
def deep_features(roi_gray_u8):
    im = cv2.resize(roi_gray_u8, (224, 224))
    x = torch.from_numpy(to_tensor_input(np.repeat(im[..., None], 3, axis=2)))[None]
    return cnn(x.to(DEVICE))[0].cpu().numpy()


def roi_features(img, mask):
    roi = crop_roi(img, mask, CONFIG["roi_pad"])
    return (extract_radiomics(roi, crop_roi(mask, mask, CONFIG["roi_pad"])),
            deep_features(roi))


def to_frames(rad_rows, deep_rows):
    rad = pd.DataFrame(rad_rows).fillna(0.0)
    deep = pd.DataFrame(np.vstack(deep_rows),
                        columns=[f"deep_{k}" for k in range(len(deep_rows[0]))])
    return rad, deep


def build_feature_table(df, use_pred_mask):
    size = CONFIG["img_size"]
    rad_rows, deep_rows, y = [], [], []
    for _, r in tqdm(df.iterrows(), total=len(df), desc="features"):
        img = load_image_gray(r["image"], size)
        mask = predict_mask(img, size) if use_pred_mask else load_merged_mask(r["masks"], size)
        if mask.sum() < 10:
            mask = np.ones_like(mask)
        rad, deep = roi_features(img, mask)
        rad_rows.append(rad)
        deep_rows.append(deep)
        y.append(int(r["class"] == "malignant"))
    return (*to_frames(rad_rows, deep_rows), np.array(y))


def cached(path, build):
    if os.path.exists(path) and not CONFIG["force_rebuild_features"]:
        d = joblib.load(path)
        print(f"loaded {path}")
    else:
        rad, deep, y = build()
        d = {"rad": rad, "deep": deep, "y": y}
        joblib.dump(d, path)
    return d["rad"], d["deep"], d["y"]

In [ ]:
mode = "pred" if CONFIG["internal_use_pred_mask"] else "gt"
rad_all, deep_all, y_all = cached(
    os.path.join(RES, f"features_internal_{mode}.joblib"),
    lambda: build_feature_table(seg_df, use_pred_mask=CONFIG["internal_use_pred_mask"]))
print(f"n={len(y_all)}  radiomics={rad_all.shape[1]}  deep={deep_all.shape[1]}  "
      f"malignant={y_all.mean():.3f}")

## 7. Classifier and statistics

Each feature set goes through `StandardScaler → SelectFromModel(LassoCV) → LogisticRegression`.
AUCs are compared with the DeLong test (fast implementation, Sun & Xu 2014) and reported
with 1000-sample bootstrap 95% CIs.

In [ ]:
ORDER = ["radiomics", "cnn", "fusion"]


def make_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("select", SelectFromModel(LassoCV(cv=5, random_state=SEED, max_iter=5000))),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])


def metrics_at(y, p, thr=0.5):
    tn, fp, fn, tp = confusion_matrix(y, (np.asarray(p) >= thr).astype(int),
                                      labels=[0, 1]).ravel()
    return {"AUC": roc_auc_score(y, p), "sens": tp / (tp + fn + 1e-9),
            "spec": tn / (tn + fp + 1e-9), "acc": (tp + tn) / len(y)}


def _midrank(x):
    order = np.argsort(x)
    z = x[order]
    n = len(x)
    t = np.zeros(n)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n)
    out[order] = t
    return out


def delong_p(y, p1, p2):
    y = np.asarray(y)
    order = (-y).argsort()
    m = int(y.sum())
    preds = np.vstack([np.asarray(p1)[order], np.asarray(p2)[order]])
    n = preds.shape[1] - m
    tx = np.array([_midrank(r[:m]) for r in preds])
    ty = np.array([_midrank(r[m:]) for r in preds])
    tz = np.array([_midrank(r) for r in preds])
    aucs = (tz[:, :m].sum(1) / m - (m + 1) / 2.0) / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    contrast = np.array([1.0, -1.0])
    z = (aucs[0] - aucs[1]) / np.sqrt(contrast @ cov @ contrast + 1e-12)
    return float(2 * norm.sf(abs(z)))


def bootstrap_auc_ci(y, p, n=1000, seed=SEED):
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) == 2:
            aucs.append(roc_auc_score(y[idx], p[idx]))
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def feature_sets_of(rad, deep):
    return {"radiomics": rad, "cnn": deep, "fusion": pd.concat([rad, deep], axis=1)}


def cross_validate(sets, y):
    skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=SEED)
    oof = {n: np.zeros(len(y)) for n in sets}
    per_fold = []
    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y)):
        for name, X in sets.items():
            p = make_model().fit(X.values[tr], y[tr]).predict_proba(X.values[te])[:, 1]
            oof[name][te] = p
            per_fold.append({**metrics_at(y[te], p), "model": name, "fold": fold})
    return oof, pd.DataFrame(per_fold)

## 8. Internal cross-validation

Stratified 5-fold CV with a fresh pipeline per fold. LassoCV runs its own inner CV, so
feature selection is nested. DeLong is computed on pooled out-of-fold predictions.

In [ ]:
feature_sets = feature_sets_of(rad_all, deep_all)
oof, cv_results = cross_validate(feature_sets, y_all)

agg = cv_results.groupby("model")[["AUC", "sens", "spec", "acc"]].agg(["mean", "std"])
cv_summary = pd.DataFrame({
    k: agg[k]["mean"].map("{:.3f}".format) + " ± " + agg[k]["std"].map("{:.3f}".format)
    for k in ["AUC", "sens", "spec", "acc"]}).reindex(ORDER).rename_axis("model").reset_index()

print(cv_summary.to_string(index=False))
print(f"\nDeLong (OOF)  fusion vs cnn: p={delong_p(y_all, oof['fusion'], oof['cnn']):.4f}  "
      f"fusion vs radiomics: p={delong_p(y_all, oof['fusion'], oof['radiomics']):.4f}")

### Reference-mask ablation

Same CV with ROIs from the manual masks. The gap to the predicted-mask arm quantifies how
much performance depends on segmentation quality.

In [ ]:
rad_ref, deep_ref, y_ref = build_feature_table(seg_df, use_pred_mask=False)
assert (y_ref == y_all).all()
oof_ref, _ = cross_validate(feature_sets_of(rad_ref, deep_ref), y_ref)

ablation = pd.DataFrame([
    {"model": n,
     "reference_mask": roc_auc_score(y_ref, oof_ref[n]),
     "predicted_mask": roc_auc_score(y_all, oof[n])} for n in ORDER])
ablation["delta"] = ablation["reference_mask"] - ablation["predicted_mask"]
ablation.round(3).to_csv(os.path.join(RES, "reference_mask_ablation.csv"), index=False)
np.save(os.path.join(RES, "oof_refmask.npy"), np.vstack([oof_ref[n] for n in ORDER]))
print(ablation.round(3).to_string(index=False))

if "shp_area" in rad_all:
    n_empty = int((rad_all["shp_area"] >= CONFIG["img_size"] ** 2).sum())
    print(f"\nempty predicted masks (full-image fallback): {n_empty}/{len(rad_all)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in oof.items():
    fpr, tpr, _ = roc_curve(y_all, p)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_all, p):.3f})")
ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set(title="ROC (out-of-fold)", xlabel="FPR", ylabel="TPR")
ax[0].legend()
frac_pos, mean_pred = calibration_curve(y_all, oof["fusion"], n_bins=8)
ax[1].plot(mean_pred, frac_pos, "o-")
ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set(title="Calibration: fusion (OOF)", xlabel="predicted", ylabel="observed")
plt.tight_layout()
plt.savefig(os.path.join(RES, "roc_calibration_oof.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Final models

Refit on all internal samples; used for SHAP and the external test.

In [ ]:
models = {name: (make_model().fit(X.values, y_all), list(X.columns))
          for name, X in feature_sets.items()}

## 10. External validation (BUS-BRA)

Different scanners and institution; masks are predicted by the BUSI-trained segmenter.
Files are matched to `bus_data.csv` by normalised id (`bus_0001-l.png` ↔ `0001-l`).

In [ ]:
def keyify(name):
    s = os.path.splitext(os.path.basename(str(name)))[0].lower()
    return re.sub(r"^(bus|mask|image|img)[_-]?", "", s)


def is_mask(p):
    return "mask" in os.path.basename(p).lower() or f"{os.sep}mask" in p.lower()


bb_root = CONFIG["busbra_root"]
files = [p for p in glob.glob(os.path.join(bb_root, "**", "*.png"), recursive=True)
         if "__MACOSX" not in p]
img_map = {keyify(p): p for p in files if not is_mask(p)}
mask_map = {keyify(p): p for p in files if is_mask(p)}

bb = pd.read_csv(glob.glob(os.path.join(bb_root, "**", "bus_data.csv"), recursive=True)[0])
bb["_key"] = bb[CONFIG["busbra_id_col"]].astype(str).map(keyify)
bb["y"] = bb[CONFIG["busbra_label_col"]].str.lower().map({"benign": 0, "malignant": 1})
bb = bb.dropna(subset=["y"])
bb = bb[bb["_key"].isin(img_map)]
print(f"BUS-BRA: {len(bb)} labelled images "
      f"({int(bb['y'].sum())} malignant), {len(mask_map)} masks")


def build_external_table():
    size = CONFIG["img_size"]
    rad_rows, deep_rows = [], []
    for key in tqdm(bb["_key"], desc="BUS-BRA"):
        img = load_image_gray(img_map[key], size)
        mp = mask_map.get(key)
        use_pred = CONFIG["ext_use_pred_mask"] or mp is None
        mask = predict_mask(img, size) if use_pred else load_merged_mask([mp], size)
        if mask.sum() < 10:
            mask = predict_mask(img, size)
        if mask.sum() < 10:
            mask = np.ones_like(mask)
        rad, deep = roi_features(img, mask)
        rad_rows.append(rad)
        deep_rows.append(deep)
    return (*to_frames(rad_rows, deep_rows), bb["y"].astype(int).to_numpy())


emode = "pred" if CONFIG["ext_use_pred_mask"] else "gt"
rad_ext, deep_ext, y_ext = cached(
    os.path.join(RES, f"features_external_{emode}.joblib"), build_external_table)

ext_sets = feature_sets_of(rad_ext, deep_ext)
ext_probs = {name: m.predict_proba(ext_sets[name].reindex(columns=cols, fill_value=0.0)
                                   .values)[:, 1]
             for name, (m, cols) in models.items()}
external_df = pd.DataFrame([{"model": n, **metrics_at(y_ext, ext_probs[n])} for n in ORDER])
external_df.round(3).to_csv(os.path.join(RES, "external_busbra_results.csv"), index=False)

print(external_df.round(3).to_string(index=False))
print(f"\nDeLong (external)  "
      f"fusion vs radiomics: p={delong_p(y_ext, ext_probs['fusion'], ext_probs['radiomics']):.4f}  "
      f"fusion vs cnn: p={delong_p(y_ext, ext_probs['fusion'], ext_probs['cnn']):.4f}")

## 11. Explainability

`LinearExplainer` gives exact SHAP values for the logistic head on the LASSO-selected,
standardised features.

In [ ]:
def linear_shap(model, columns, X):
    pre = Pipeline(model.steps[:-1])
    X_t = pre.transform(X.values)
    selected = list(np.array(columns)[model.named_steps["select"].get_support()])
    values = shap.LinearExplainer(model.named_steps["clf"], X_t)(X_t).values
    return values, X_t, selected


shap_vals, X_fus_t, selected = linear_shap(*models["fusion"], feature_sets["fusion"])
print(f"fusion: {len(selected)} features selected by LASSO")
shap.summary_plot(shap_vals, features=X_fus_t, feature_names=selected,
                  max_display=len(selected), show=False)
plt.tight_layout()
plt.savefig(os.path.join(RES, "shap_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

imp = pd.DataFrame({"feature": selected, "mean_abs_shap": np.abs(shap_vals).mean(0)})
imp["modality"] = np.where(imp["feature"].str.startswith("deep_"), "CNN", "radiomics")
imp = imp.sort_values("mean_abs_shap", ascending=False)
imp.to_csv(os.path.join(RES, "shap_feature_importance.csv"), index=False)

contrib = imp.groupby("modality")["mean_abs_shap"].sum()
print(contrib.round(4).to_string())
ax = contrib.plot(kind="bar", rot=0, title="Fusion model: contribution by modality")
ax.set_ylabel("sum of mean |SHAP|")
plt.tight_layout()
plt.savefig(os.path.join(RES, "modality_contribution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sv_r, Xr_t, sel_r = linear_shap(*models["radiomics"], rad_all)
print(f"radiomics: {len(sel_r)} features selected")
shap.summary_plot(sv_r, features=Xr_t, feature_names=sel_r, max_display=len(sel_r), show=False)
plt.tight_layout()
plt.savefig(os.path.join(RES, "shap_radiomics.png"), dpi=150, bbox_inches="tight")
plt.show()

Deep features have no semantic names. Each of the top-10 is described by the radiomics
features it correlates with most; low max |r| (< ~0.3) indicates information the
handcrafted set does not capture.

In [ ]:
fused = feature_sets["fusion"]
rad_cols = list(rad_all.columns)
proxy_rows = []
for d in [f for f in imp["feature"] if f.startswith("deep_")][:10]:
    r = fused[rad_cols].corrwith(fused[d]).dropna()
    top3 = r.abs().sort_values(ascending=False).index[:3]
    proxy_rows.append({"deep_feature": d,
                       "behaves_like (radiomics, signed r)":
                           ", ".join(f"{k} ({r[k]:+.2f})" for k in top3),
                       "max_|r|": round(float(r.abs().max()), 2)})
deep_proxy = pd.DataFrame(proxy_rows)
deep_proxy.to_csv(os.path.join(RES, "deep_feature_proxies.csv"), index=False)
deep_proxy

## 12. Figures

Vector PDF + 300 dpi PNG.

In [ ]:
plt.rcParams.update({
    "savefig.dpi": 300, "figure.dpi": 110, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
    "font.family": "DejaVu Sans",
})
PALETTE = {"radiomics": "#4C72B0", "cnn": "#DD8452", "fusion": "#55A868"}


def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(RES, f"{name}.{ext}"))


ci_df = pd.DataFrame(
    [{"model": m, "set": s, "auc": roc_auc_score(y, p[m]),
      **dict(zip(("lo", "hi"), bootstrap_auc_ci(y, p[m])))}
     for s, y, p in [("internal", y_all, oof), ("external", y_ext, ext_probs)]
     for m in ORDER])
ci_df.to_csv(os.path.join(RES, "auc_bootstrap_ci.csv"), index=False)
ci_df.round(3)

In [ ]:
idxs = list(test_idx)[:4]
fig, axes = plt.subplots(len(idxs), 3, figsize=(7.5, 2.4 * len(idxs)))
for r, i in enumerate(idxs):
    row = seg_df.loc[i]
    img = load_image_gray(row["image"], CONFIG["img_size"])
    gt = load_merged_mask(row["masks"], CONFIG["img_size"])
    pr = predict_mask(img, CONFIG["img_size"])
    panels = [(None, "Image", None), (gt, "Ground truth", "#55A868"),
              (pr, "Prediction", "#C44E52")]
    for c, (overlay, title, color) in enumerate(panels):
        a = axes[r, c]
        a.imshow(img, cmap="gray")
        if overlay is not None:
            a.imshow(np.ma.masked_where(overlay == 0, overlay),
                     cmap=mpl.colors.ListedColormap([color]), alpha=0.45)
        a.axis("off")
        if r == 0:
            a.set_title(title)
    axes[r, 2].text(6, 20, f"Dice={dice_iou(gt, pr)[0]:.2f}", color="white", fontsize=10,
                    bbox=dict(facecolor="black", pad=1, edgecolor="none"))
plt.tight_layout()
savefig(fig, "fig_segmentation_examples")
plt.show()

if seg_history:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(1, len(seg_history) + 1), seg_history, marker="o", color=PALETTE["fusion"])
    ax.set(xlabel="epoch", ylabel="validation Dice", title="Segmentation training")
    savefig(fig, "fig_seg_training_curve")
    plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot(test_dice, widths=0.5, patch_artist=True,
              boxprops=dict(facecolor=PALETTE["cnn"], alpha=0.6),
              medianprops=dict(color="black"))
ax[0].scatter(np.random.normal(1, 0.04, len(test_dice)), test_dice, s=18, color="#333",
              alpha=0.5)
ax[0].set_xticks([1], ["test set"])
ax[0].set(ylabel="Dice", ylim=(-0.02, 1.02),
          title=f"Per-image Dice (median={np.median(test_dice):.2f})")
ax[1].hist(test_dice, bins=np.linspace(0, 1, 21), color=PALETTE["cnn"], edgecolor="white")
ax[1].set(xlabel="Dice", ylabel="# images", title="Dice distribution")
plt.tight_layout()
savefig(fig, "fig_dice_distribution")
plt.show()

In [ ]:
agg_auc = cv_results.groupby("model")["AUC"].agg(["mean", "std"]).reindex(ORDER)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(ORDER, agg_auc["mean"], yerr=agg_auc["std"], capsize=6,
              color=[PALETTE[m] for m in ORDER])
for b, mu in zip(bars, agg_auc["mean"]):
    ax.text(b.get_x() + b.get_width() / 2, mu + 0.01, f"{mu:.3f}", ha="center", fontsize=10)
ax.set(ylabel=f"AUC ({CONFIG['n_folds']}-fold CV)", ylim=(0.5, 1.0),
       title="Internal cross-validation")
ax.grid(axis="x")
savefig(fig, "fig_cv_auc")
plt.show()

x = np.arange(len(ORDER))
w = 0.38
fig, ax = plt.subplots(figsize=(7, 4.5))
for s, off, col in [("internal", -w / 2, "#4C72B0"), ("external", w / 2, "#C44E52")]:
    sub = ci_df[ci_df["set"] == s].set_index("model").reindex(ORDER)
    err = np.vstack([sub["auc"] - sub["lo"], sub["hi"] - sub["auc"]])
    ax.bar(x + off, sub["auc"], w, yerr=err, capsize=5, label=s.capitalize(), color=col)
ax.axhline(0.5, ls="--", c="gray", lw=1)
ax.set_xticks(x, ORDER)
ax.set(ylabel="AUC (95% CI)", ylim=(0.45, 1.0), title="Internal vs external generalization")
ax.legend()
ax.grid(axis="x")
savefig(fig, "fig_internal_vs_external")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for a, title, y, probs in [(axes[0], "Internal (out-of-fold)", y_all, oof),
                           (axes[1], "External (BUS-BRA)", y_ext, ext_probs)]:
    for m in ORDER:
        fpr, tpr, _ = roc_curve(y, probs[m])
        a.plot(fpr, tpr, color=PALETTE[m], lw=2,
               label=f"{m} ({roc_auc_score(y, probs[m]):.3f})")
    a.plot([0, 1], [0, 1], "k--", lw=1)
    a.set(title=title, xlabel="1 - specificity", ylabel="sensitivity")
    a.legend()
plt.tight_layout()
savefig(fig, "fig_roc_panels")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4))
for a, title, y, p in [(axes[0], "Internal (OOF)", y_all, oof["fusion"]),
                       (axes[1], "External", y_ext, ext_probs["fusion"])]:
    cm = confusion_matrix(y, (p >= 0.5).astype(int), labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["benign", "malignant"]).plot(
        ax=a, colorbar=False, cmap="Blues")
    a.set_title(f"Fusion: {title}")
    a.grid(False)
plt.tight_layout()
savefig(fig, "fig_confusion_fusion")
plt.show()

top = imp.head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["feature"], top["mean_abs_shap"],
        color=[PALETTE["cnn" if f.startswith("deep_") else "radiomics"] for f in top["feature"]])
ax.set(xlabel="mean |SHAP|", title="Top features (blue = radiomics, orange = CNN)")
ax.grid(axis="y")
savefig(fig, "fig_top_features")
plt.show()

## 13. Artifacts

In [ ]:
def describe(feat):
    f = feat.lower()
    if f.startswith("deep_"):
        return ("CNN deep feature", f"ResNet-50 pooled activation #{feat.split('_', 1)[1]}")
    if f.startswith("shp_") or "shape" in f:
        return ("Radiomics: shape", "lesion geometry: size, compactness, elongation, margin")
    if f.startswith("fo_") or "firstorder" in f:
        return ("Radiomics: first order", "intensity-histogram statistic within the lesion")
    if any(t in f for t in ["glcm", "glrlm", "glszm", "gldm", "ngtdm"]):
        return ("Radiomics: texture", "grey-level texture / heterogeneity")
    return ("Radiomics: other", "")


pd.DataFrame([(f, *describe(f)) for f in fused.columns],
             columns=["feature", "type", "description"]).to_csv(
    os.path.join(RES, "feature_dictionary.csv"), index=False)

fused.assign(label=y_all).to_csv(os.path.join(RES, "features_all.csv"), index=False)
cv_summary.to_csv(os.path.join(RES, "cv_summary.csv"), index=False)
cv_results.to_csv(os.path.join(RES, "cv_results_per_fold.csv"), index=False)
np.save(os.path.join(RES, "shap_values.npy"), shap_vals)
for name, (m, cols) in models.items():
    joblib.dump({"model": m, "columns": cols}, os.path.join(RES, f"model_{name}.joblib"))
with open(os.path.join(RES, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

print(sorted(os.listdir(RES)))

In [ ]:
def ver(mod):
    return getattr(mod, "__version__", "?")


env = {"python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
       "smp": ver(smp), "sklearn": ver(sklearn),
       "skimage": ver(skimage), "opencv": cv2.__version__,
       "imagehash": ver(imagehash), "shap": ver(shap),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
env

## Limitations

- BUSI lacks patient IDs; leakage is mitigated by pHash dedup but splits remain image-level.
- The segmenter and the internal classifiers share BUSI images, so internal predicted-mask
  results may be mildly optimistic. BUS-BRA is the clean generalisation estimate.
- With the scikit-image backend the radiomics set is 18 features. Install PyRadiomics for the
  full IBSI-compliant set.